# 03 - Preprocessing and Feature Preparation

**Owner:** Aliyan

This notebook creates leakage-safe, reproducible representations for category
classification and research-paper discovery. It validates the cleaned dataset,
encodes the target, creates a stratified split, fits transformations on the
training partition only, and saves split metadata and feature matrices.

In [1]:
# Data-path cell: Kaggle path first, local clone fallback (run this first)
from pathlib import Path
import os
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

TEST_SIZE = 0.20
REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
from src.config import CATEGORIES, SEED, EMBEDDING_MODEL, SAMPLE_SIZE

EMBEDDING_SAMPLE_SIZE = SAMPLE_SIZE
ENV_DATA_PATH = "RESEARCH_PAPERS_PATH"
DATA_CANDIDATES = []
if os.environ.get(ENV_DATA_PATH):
    DATA_CANDIDATES.append(Path(os.environ[ENV_DATA_PATH]).expanduser())
DATA_CANDIDATES.extend([
    Path("/kaggle/input/arxiv-papers-cleaned/papers.parquet"),
    REPO_DIR / "data" / "papers.parquet",
])
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if DATA_PATH is None:
    checked = "\n".join(f"- {path}" for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        "Could not find the cleaned papers.parquet dataset. Checked:\n" + checked
    )
papers = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(papers):,} papers from {DATA_PATH}")

Loaded 119,919 papers from /media/ibraheem/HDD/research_paper_discovery/data/papers.parquet


## Reproducibility contract

- Input: `data/papers.parquet` produced by the cleaning notebook (Kaggle path is checked first).
- Target: `primary_category`, encoded with a seeded-free `LabelEncoder`; the class mapping is saved to `results/label_classes.json`.
- Text: `text_lemma` for TF-IDF and `text_raw` for sentence-transformer embeddings.
- Numeric metadata: `year`, `month`, and abstract/title word counts.
- Split: stratified 80/20 train/test split with the shared seed (42).
- All learned transformations are fitted on the training partition only.
- Outputs are written to `results/` and use the same row order as the saved ids.
- Full-corpus embeddings are generated separately on Kaggle/GPU by `src/embed_all.py`; the subsection below is marked Kaggle/GPU only.

## Text normalization already done

Text normalization was performed in `notebooks/01_data_cleaning.ipynb`:
`text_lemma` is lowercased, stop-word-free and spaCy-lemmatized, with generic
academic words removed. TF-IDF below uses `text_lemma`, so no lowercasing or
lemmatization is repeated here. `text_raw` keeps case and punctuation for the
embedding model. The numeric features below are the only features that need
additional scaling.

In [2]:
required = {"id", "text_raw", "text_lemma", "primary_category", "year", "month", "title", "abstract"}
missing = required.difference(papers.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
if papers["id"].duplicated().any():
    raise ValueError("Input ids must be unique")
if papers[["text_raw", "text_lemma", "primary_category"]].isna().any().any():
    raise ValueError("Text and target columns must not contain nulls")
if not papers["primary_category"].isin(CATEGORIES).all():
    unexpected = sorted(set(papers["primary_category"]) - set(CATEGORIES))
    raise ValueError(f"Unexpected labels: {unexpected}; expected only {CATEGORIES}")
print("Target distribution (raw labels):")
display(papers["primary_category"].value_counts().rename("count").to_frame())

Target distribution (raw labels):


,count
primary_category,
cs.CL,39980
cs.CV,39973
cs.LG,39966


## Target encoding

`LabelEncoder` maps the three target strings to fixed integer class ids. Fitting
it on the full label column is safe because it only defines a naming mapping;
no feature information is learned. The mapping and the class distribution are
saved so Phase 2 can recover the original labels.

In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder().fit(papers["primary_category"])
class_counts = papers["primary_category"].value_counts().reindex(label_encoder.classes_)
label_classes = {
    "classes_": label_encoder.classes_.tolist(),
    "counts": {str(c): int(class_counts[c]) for c in label_encoder.classes_},
}
(RESULTS_DIR / "label_classes.json").write_text(
    json.dumps(label_classes, indent=2) + "\n", encoding="utf-8"
)
print("classes_:", label_encoder.classes_.tolist())
display(class_counts.rename("count").to_frame())

classes_: ['cs.CL', 'cs.CV', 'cs.LG']


,count
primary_category,
cs.CL,39980
cs.CV,39973
cs.LG,39966


## Stratified train/test split

The split uses the shared seed and preserves the class proportions. The saved
`split_metadata.json` records train/test ids; encoded label arrays are written
alongside the string label arrays for Phase 2.

In [4]:
indices = np.arange(len(papers))
train_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=SEED,
    stratify=papers["primary_category"]
)
train = papers.iloc[train_idx].copy()
test = papers.iloc[test_idx].copy()
train_ids = set(train["id"])
test_ids = set(test["id"])
if train_ids.intersection(test_ids):
    raise ValueError("Train and test IDs must be disjoint")
if len(train) + len(test) != len(papers):
    raise ValueError("Train and test rows must cover the input exactly once")
print(f"train={len(train):,}, test={len(test):,}")
print("train/test proportions:")
display(pd.concat([
    train["primary_category"].value_counts(normalize=True).rename("train"),
    test["primary_category"].value_counts(normalize=True).rename("test")
], axis=1))

split_metadata = {
    "seed": SEED,
    "test_size": TEST_SIZE,
    "train_ids": train["id"].tolist(),
    "test_ids": test["id"].tolist(),
}
(RESULTS_DIR / "split_metadata.json").write_text(
    json.dumps(split_metadata, indent=2), encoding="utf-8"
)

train=95,935, test=23,984
train/test proportions:


,train,test
primary_category,,
cs.CL,0.333392,0.333389
cs.CV,0.333330,0.333347
cs.LG,0.333278,0.333264


2158619

## TF-IDF word features

The vocabulary and inverse-document-frequency weights are learned from training
text only. The resulting matrices are L2-normalized by the vectorizer
(`norm="l2"`), so each row has unit length.

In [5]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_df=0.95,
    sublinear_tf=True, norm="l2", dtype=np.float32
)
X_train_tfidf = tfidf.fit_transform(train["text_lemma"])
X_test_tfidf = tfidf.transform(test["text_lemma"])
y_train = train["primary_category"].to_numpy()
y_test = test["primary_category"].to_numpy()
y_train_enc = label_encoder.transform(train["primary_category"])
y_test_enc = label_encoder.transform(test["primary_category"])
np.save(RESULTS_DIR / "y_train_enc.npy", y_train_enc)
np.save(RESULTS_DIR / "y_test_enc.npy", y_test_enc)
if X_train_tfidf.shape[0] != len(y_train) or X_test_tfidf.shape[0] != len(y_test):
    raise ValueError("TF-IDF row counts must match label row counts")
print("TF-IDF shapes:", X_train_tfidf.shape, X_test_tfidf.shape)
from scipy.sparse import save_npz
save_npz(RESULTS_DIR / "tfidf_train.npz", X_train_tfidf)
save_npz(RESULTS_DIR / "tfidf_test.npz", X_test_tfidf)
np.save(RESULTS_DIR / "y_train.npy", y_train)
np.save(RESULTS_DIR / "y_test.npy", y_test)

TF-IDF shapes: (95935, 919475) (23984, 919475)


## Top TF-IDF terms per class

For each class, we average the L2-normalized TF-IDF vectors of its training
rows and print the 15 highest-weight terms. This shows which lemmatized words
characterize each category according to the training data only.

In [6]:
feature_names = tfidf.get_feature_names_out()
for category in CATEGORIES:
    mask = y_train == category
    mean_tfidf = np.asarray(X_train_tfidf[mask].mean(axis=0)).ravel()
    top_idx = mean_tfidf.argsort()[-15:][::-1]
    top_terms = [(feature_names[i], round(float(mean_tfidf[i]), 4)) for i in top_idx]
    print(f"{category}: {top_terms}")
print("Note: every row of X_train_tfidf and X_test_tfidf is L2-normalized (norm='l2').")

cs.LG: [('learning', 0.0196), ('model', 0.0166), ('datum', 0.0147), ('algorithm', 0.0141), ('network', 0.0137), ('learn', 0.0131), ('neural', 0.0125), ('base', 0.0111), ('neural network', 0.0104), ('deep', 0.0103), ('training', 0.0102), ('graph', 0.0098), ('time', 0.0098), ('machine', 0.0093), ('function', 0.0087)]


cs.CL: [('language', 0.0276), ('model', 0.0226), ('text', 0.0157), ('language model', 0.0148), ('large', 0.0127), ('llms', 0.0119), ('dataset', 0.0118), ('base', 0.0113), ('word', 0.011), ('large language', 0.0107), ('datum', 0.0102), ('system', 0.0098), ('llm', 0.0096), ('knowledge', 0.0096), ('human', 0.0094)]


cs.CV: [('image', 0.0272), ('model', 0.0151), ('object', 0.0149), ('feature', 0.014), ('dataset', 0.0135), ('network', 0.0134), ('base', 0.0128), ('video', 0.0119), ('detection', 0.0116), ('segmentation', 0.0113), ('visual', 0.0104), ('learning', 0.0104), ('datum', 0.0098), ('training', 0.0095), ('deep', 0.0094)]
Note: every row of X_train_tfidf and X_test_tfidf is L2-normalized (norm='l2').


## Scaling numeric metadata and when scaling matters

- Tree models (decision trees, random forests, gradient boosting) split on raw thresholds and do **not** need feature scaling.
- Distance/kernel/gradient models such as Logistic Regression, SVM and kNN **do** benefit from (or require) standardized numeric features, so `StandardScaler` is fitted on the training rows only.
- Multinomial Naive Bayes expects non-negative inputs; do **not** feed it `StandardScaler` output (which can be negative). Use raw counts or TF-IDF instead.
- TF-IDF rows are already L2-normalized by the vectorizer, so no additional scaling is applied to them.
- The sentence embeddings produced below and by `src/embed_all.py` are L2-normalized by `normalize_embeddings=True`.

In [7]:
def word_count(series):
    return series.astype(str).str.split().str.len().to_numpy(dtype=np.float32)

numeric_columns = ["year", "month", "title_words", "abstract_words"]
def numeric_features(frame):
    return np.column_stack([
        frame["year"].to_numpy(dtype=np.float32),
        frame["month"].to_numpy(dtype=np.float32),
        word_count(frame["title"]),
        word_count(frame["abstract"]),
    ])

scaler = StandardScaler()
X_train_numeric = scaler.fit_transform(numeric_features(train)).astype(np.float32)
X_test_numeric = scaler.transform(numeric_features(test)).astype(np.float32)
if np.isnan(X_train_numeric).any() or np.isnan(X_test_numeric).any():
    raise ValueError("Numeric features must not contain NaNs")
if X_train_numeric.shape[0] != len(y_train) or X_test_numeric.shape[0] != len(y_test):
    raise ValueError("Numeric row counts must match label row counts")
np.save(RESULTS_DIR / "numeric_train.npy", X_train_numeric)
np.save(RESULTS_DIR / "numeric_test.npy", X_test_numeric)
print("Numeric shapes:", X_train_numeric.shape, X_test_numeric.shape)

Numeric shapes: (95935, 4) (23984, 4)


## Quick demo: sentence-transformer embeddings (CPU-safe subset)

This 10,000-row stratified subset keeps local execution practical. It is a
**demo only**. The full-corpus embeddings (all 119,919 rows) are produced by
`src/embed_all.py` on Kaggle/GPU in the next section.

In [8]:
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

embedding_n = min(EMBEDDING_SAMPLE_SIZE, len(papers))
embedding_train_n = min(round(embedding_n * len(train) / len(papers)), len(train))
embedding_test_n = min(embedding_n - embedding_train_n, len(test))
embedding_train_idx, _ = train_test_split(
    np.arange(len(train)), train_size=embedding_train_n, random_state=SEED,
    stratify=train["primary_category"]
)
embedding_test_idx, _ = train_test_split(
    np.arange(len(test)), train_size=embedding_test_n, random_state=SEED,
    stratify=test["primary_category"]
)
embedding_train = train.iloc[embedding_train_idx].sort_values("id").reset_index(drop=True)
embedding_test = test.iloc[embedding_test_idx].sort_values("id").reset_index(drop=True)
encoder = SentenceTransformer(EMBEDDING_MODEL)
embeddings_train = encoder.encode(
    embedding_train["text_raw"].tolist(), batch_size=32,
    show_progress_bar=True, normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)
embeddings_test = encoder.encode(
    embedding_test["text_raw"].tolist(), batch_size=32,
    show_progress_bar=True, normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)
np.save(RESULTS_DIR / "embeddings_train.npy", embeddings_train)
np.save(RESULTS_DIR / "embeddings_test.npy", embeddings_test)
embedding_train[["id", "primary_category"]].assign(split="train").to_parquet(
    RESULTS_DIR / "embedding_ids_train.parquet", index=False
)
embedding_test[["id", "primary_category"]].assign(split="test").to_parquet(
    RESULTS_DIR / "embedding_ids_test.parquet", index=False
)
for split, matrix, metadata in [
    ("train", embeddings_train, embedding_train),
    ("test", embeddings_test, embedding_test),
]:
    if matrix.shape[0] != len(metadata) or metadata["id"].duplicated().any():
        raise ValueError(f"Embedding IDs do not match {split} embedding rows")
print("Embedding shapes:", embeddings_train.shape, embeddings_test.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Embedding shapes: (8000, 384) (2000, 384)


## Kaggle/GPU only: full-corpus embeddings (do not run locally)

The full 119,919-row embedding job is intentionally **not** run in this
notebook locally. Run it on Kaggle with a GPU and the `arxiv-papers-cleaned`
dataset attached:

```bash
python src/embed_all.py \
  --papers /kaggle/input/arxiv-papers-cleaned/papers.parquet \
  --out /kaggle/working/embeddings.npy \
  --check results/embeddings_check.json
```

The guarded cell below is left with `RUN_FULL_EMBEDDINGS = False` so this
notebook can execute locally without downloading the model or running the
heavy job. Set it to `True` only on a Kaggle GPU kernel.

In [9]:
RUN_FULL_EMBEDDINGS = False
if not RUN_FULL_EMBEDDINGS:
    print("SKIPPED: full-corpus embeddings are Kaggle/GPU only. "
          "Run src/embed_all.py and publish embeddings.npy as a Kaggle Dataset.")
else:
    import subprocess, sys
    subprocess.run([
        sys.executable, str(REPO_DIR / "src" / "embed_all.py"),
        "--papers", "/kaggle/input/arxiv-papers-cleaned/papers.parquet",
        "--out", "/kaggle/working/embeddings.npy",
        "--check", str(RESULTS_DIR / "embeddings_check.json"),
    ], check=True)

SKIPPED: full-corpus embeddings are Kaggle/GPU only. Run src/embed_all.py and publish embeddings.npy as a Kaggle Dataset.


## Feature manifest

The manifest is written from the actual in-memory objects above (including the
true TF-IDF feature count), so it cannot drift away from the executed cells.

In [10]:
manifest = {
    "seed": SEED,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_sample_size": embedding_n,
    "embedding_train_rows": len(embedding_train),
    "embedding_test_rows": len(embedding_test),
    "embedding_files": {
        "train": "embeddings_train.npy",
        "test": "embeddings_test.npy",
        "train_ids": "embedding_ids_train.parquet",
        "test_ids": "embedding_ids_test.parquet"
    },
    "tfidf_ngram_range": [1, 2],
    "tfidf_norm": "l2",
    "label_classes_file": "label_classes.json",
    "label_classes": label_encoder.classes_.tolist(),
    "numeric_columns": numeric_columns,
    "train_rows": len(train),
    "test_rows": len(test),
    "tfidf_features": X_train_tfidf.shape[1],
    "embedding_dimension": int(embeddings_train.shape[1]),
}
(RESULTS_DIR / "feature_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
assert manifest["tfidf_features"] == X_train_tfidf.shape[1]
assert manifest["train_rows"] == len(train) and manifest["test_rows"] == len(test)
print(json.dumps(manifest, indent=2))

{
  "seed": 42,
  "embedding_model": "all-MiniLM-L6-v2",
  "embedding_sample_size": 10000,
  "embedding_train_rows": 8000,
  "embedding_test_rows": 2000,
  "embedding_files": {
    "train": "embeddings_train.npy",
    "test": "embeddings_test.npy",
    "train_ids": "embedding_ids_train.parquet",
    "test_ids": "embedding_ids_test.parquet"
  },
  "tfidf_ngram_range": [
    1,
    2
  ],
  "tfidf_norm": "l2",
  "label_classes_file": "label_classes.json",
  "label_classes": [
    "cs.CL",
    "cs.CV",
    "cs.LG"
  ],
  "numeric_columns": [
    "year",
    "month",
    "title_words",
    "abstract_words"
  ],
  "train_rows": 95935,
  "test_rows": 23984,
  "tfidf_features": 919475,
  "embedding_dimension": 384
}
